In [0]:
import logging

In [0]:
logger = logging.getLogger("DatabricksWorkflow")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
if not logger.hasHandlers():
    logger.addHandler(handler)

In [0]:
config_path = "dbfs:/configs/config.json"
try:
    config = spark.read.option("multiline", "true").json(config_path)
    logger.info(f"Successfully read config file from {config_path}")
except Exception as e:
    logger.error(f"Could not read config file at {config_path}: {e}", exc_info=True)
    raise FileNotFoundError(f"Could not read config file at {config_path}: {e}")

try:
    first_row = config.first()
    env = first_row["env"].strip().lower()
    lz_key = first_row["lz_key"].strip().lower()
    logger.info(f"Extracted configs: env={env}, lz_key={lz_key}")
except Exception as e:
    logger.error(f"Missing expected keys 'env' or 'lz_key' in config file: {e}", exc_info=True)
    raise KeyError(f"Missing expected keys 'env' or 'lz_key' in config file: {e}")

try:
    keyvault_name = f"ingest{lz_key}-meta002-{env}"
    logger.info(f"Constructed keyvault name: {keyvault_name}")
except Exception as e:
    logger.error(f"Error constructing keyvault name: {e}", exc_info=True)
    raise ValueError(f"Error constructing keyvault name: {e}")


In [0]:

try:
    client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-SECRET from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}")

try:
    tenant_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-TENANT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}")

try:
    client_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}")

logger.info("✅ Successfully retrieved all Service Principal secrets from Key Vault")


In [0]:
# --- Parameterise containers ---
curated_storage_account = f"ingest{lz_key}curated{env}"
curated_container = "gold"
silver_curated_container = "silver"

# --- Assign OAuth to storage accounts ---
storage_accounts = [curated_storage_account]

for storage_account in storage_accounts:
    try:
        configs = {
            f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
            f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
                "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
            f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
            f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
            f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
                f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
        }

        for key, val in configs.items():
            try:
                spark.conf.set(key, val)
            except Exception as e:
                logger.error(f"Failed to set Spark config '{key}' for storage account '{storage_account}': {e}", exc_info=True)
                raise RuntimeError(f"Failed to set Spark config '{key}' for storage account '{storage_account}': {e}")

        logger.info(f"✅ Successfully configured OAuth for storage account: {storage_account}")

    except Exception as e:
        logger.error(f"Error configuring OAuth for storage account '{storage_account}': {e}", exc_info=True)
        raise RuntimeError(f"Error configuring OAuth for storage account '{storage_account}': {e}")


In [0]:
cdam_publish_payload_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/CDAM/publish_payload_audit")
cdam_publish_payload_result.createOrReplaceTempView("cdam_publish_payload_result")

cdam_call_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CDAM/ack_audit")
cdam_call_result.createOrReplaceTempView("cdam_call_result")

display(spark.sql("""
    SELECT 
        COALESCE(t2.RunId, t1.RunId) AS RunID,
        COALESCE(t2.CaseNo, t1.CaseNo) as `CaseNo`,
        t1.Status as `Case Note Document URL Publish Status`,
        t2.Status as `CDAM Call Status`,
        t1.FileName as `Case Note Document File Name`,
        t1.FileURL as `Case Note Document File URL`,
        t1.PublishingDateTime as `Case Note Document Publish Publishing Date Time`,
        t1.Error as `Case Note Document Publish Error`,
        t2.StartDateTime as `CDAM Call Function App Start Date Time`,
        t2.EndDateTime as `CDAM Call Function App End Date Time`,
        t2.Error as `CDAM Call Function App Error`,
        t2.document_filename as `CDAM document_filename`,
        t2.document_url as `CDAM document_url`,
        t2.createdOn as `CDAM createdOn`
    FROM cdam_publish_payload_result t1
        FULL OUTER JOIN cdam_call_result t2 ON t1.CaseNo = t2.CaseNo AND t1.RunID = t2.RunID
    ORDER BY `Case Note Document Publish Publishing Date Time` DESC
"""))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
active_case_note_document_segmentation = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/APPEALS/CDAM/stg_apl_combined")
active_case_note_document_segmentation.createOrReplaceTempView("active_case_note_document_segmentation")
 
active_case_note_documents_generated = spark.read.format("delta").load(f"abfss://{curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/APPEALS/CDAM/gold_appeals_with_html")
active_case_note_documents_generated.createOrReplaceTempView("active_case_note_documents_generated")

cdam_publish_payload_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/CDAM/publish_payload_audit")
cdam_publish_payload_result.createOrReplaceTempView("cdam_publish_payload_result")

cdam_call_result = spark.read.format("delta").load(f"abfss://{silver_curated_container}@{curated_storage_account}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/AUDIT/APPEALS/CDAM/ack_audit")
cdam_call_result.createOrReplaceTempView("cdam_call_result")

spark.sql("""
    SELECT 
        s.CaseNo AS CaseNo,
        CASE WHEN s.CaseNo IS NOT NULL
            THEN 'Yes'
        END AS `Case Note Document Segmentation Status`,
        CASE WHEN g.CaseNo IS NOT NULL
            THEN 'SUCCESS'
        END AS `Case Note Document Generation Status`,
        p.Status AS `Case Note Document URL Publish Status`,
        r.Status AS `CDAM Call Status`,
        p.FileName as `Case Note Document File Name`,
        p.FileUrl as `Case Note Document File URL`,
        p.PublishingDateTime as `Case Note Document Publish Publishing Date Time`,
        p.Error as `Case Note Document Publish Error`,
        r.StartDateTime as `CDAM Call Function App Start Date Time`,
        r.EndDateTime as `CDAM Call Function App End Date Time`,
        r.Error as `CDAM Call Function App Error`,
        r.document_filename as `CDAM document_filename`,
        r.document_url as `CDAM document_url`,
        r.createdOn as `CDAM createdOn`
    FROM active_case_note_document_segmentation s
        FULL OUTER JOIN active_case_note_documents_generated g ON s.CaseNo = g.CaseNo
        FULL OUTER JOIN cdam_publish_payload_result p ON s.CaseNo = p.CaseNo
        FULL OUTER JOIN cdam_call_result r ON s.CaseNo = r.CaseNo
    ORDER BY `Case Note Document Publish Publishing Date Time` DESC
""").display()

In [0]:
active_case_note_document_segmentation.groupBy("CaseNo").count().createOrReplaceTempView("deduplicated_active_case_note_documentation_segmentation")
cdam_publish_payload_result.groupBy("CaseNo").count().createOrReplaceTempView("deduplicated_cdam_publish_payload_result")
cdam_call_result.groupBy("CaseNo").count().createOrReplaceTempView("deduplicated_cdam_call_result")

tables = [
    "active_case_note_document_segmentation",
    "active_case_note_documents_generated",
    "deduplicated_active_case_note_documentation_segmentation",
    "cdam_publish_payload_result",
    "cdam_call_result",
    "deduplicated_cdam_publish_payload_result",
    "deduplicated_cdam_call_result",
]

status_tables = {"cdam_publish_payload_result", "cdam_call_result"}

row_counts = " UNION ALL ".join(
    f"SELECT {i+1} AS ind, '{t}' AS table_name, COUNT(*) AS row_count FROM {t}"
    for i, t in enumerate(tables)
)

status_pivots = " UNION ALL ".join(
    f"SELECT '{t}' AS table_name, Status, COUNT(*) AS status_count FROM {t} GROUP BY Status"
    for t in status_tables
)

spark.sql(f"""
    WITH row_counts AS ({row_counts}),
    status_counts AS ({status_pivots}),
    status_pivot AS (
        SELECT
            table_name,
            MAX(CASE WHEN Status = 'SUCCESS' THEN status_count END) AS Status_Success,
            SUM(CASE WHEN Status != 'SUCCESS' THEN status_count ELSE 0 END) AS Status_Other
        FROM status_counts
        GROUP BY table_name
    )
    SELECT
        r.ind,
        r.table_name,
        r.row_count,
        s.Status_Success,
        s.Status_Other
    FROM row_counts r
    LEFT JOIN status_pivot s ON r.table_name = s.table_name
    ORDER BY r.ind
""").display()

In [0]:
dbutils.notebook.exit("Notebook completed successfully")